# Calgary Traffic Risk Intelligence

## 01 — Data Preparation

This notebook loads, inspects, cleans, validates, and prepares the
City of Calgary Traffic Incidents and Traffic Signals datasets.

### Objectives

- Preserve the original raw datasets unchanged
- Inspect dataset structure and quality
- Standardize column names and data types
- Convert incident timestamps from UTC to Calgary local time
- Engineer temporal features for later analysis
- Clean signalized-intersection data
- Validate cleaned datasets
- Export reproducible cleaned CSV files

In [1]:
# Import the libraries used for file paths, data handling, and notebook display.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 150)

In [2]:
# Detect the project root and build paths to the raw and cleaned data folders.
cwd = Path.cwd()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

INCIDENTS_FILE = RAW_DIR / "Traffic_Incidents_20260928.csv"
SIGNALS_FILE = RAW_DIR / "Traffic_Signals_20260928.csv"

print("Project root:", PROJECT_ROOT)
print("Incidents file exists:", INCIDENTS_FILE.exists())
print("Signals file exists:", SIGNALS_FILE.exists())

missing_raw_files = [
    path for path in [INCIDENTS_FILE, SIGNALS_FILE]
    if not path.exists()
]

if missing_raw_files:
    raise FileNotFoundError(
        "Place the two City of Calgary source CSV files in data/raw/. "
        f"Missing: {missing_raw_files}"
    )

Project root: c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence
Incidents file exists: True
Signals file exists: True


## Load Raw Data

The files in `data/raw/` are treated as immutable source data.
No manual modifications are made to these files.

In [3]:
# Load both raw CSV files into pandas DataFrames without modifying the source files.
incidents_raw = pd.read_csv(INCIDENTS_FILE)
signals_raw = pd.read_csv(SIGNALS_FILE)

print(f"Traffic incidents: {incidents_raw.shape[0]:,} rows × {incidents_raw.shape[1]} columns")
print(f"Traffic signals:   {signals_raw.shape[0]:,} rows × {signals_raw.shape[1]} columns")

Traffic incidents: 64,798 rows × 10 columns
Traffic signals:   1,791 rows × 14 columns


In [4]:
# Preview a few traffic-incident records to understand the raw structure.
display(incidents_raw.head())

,INCIDENT INFO,DESCRIPTION,START_DT_UTC,MODIFIED_DT_UTC,QUADRANT,Longitude,Latitude,Count,id,Point
0,9 Avenue and 16 Street SE,Traffic incident.,2026/09/18 09:36:31 PM,2026/09/18 09:39:16 PM,SE,-114.022152,51.036435,1.0,2026-09-18 21:36:31+00:0051.03643495929764-114.02215204531316,POINT (-114.02215204531316 51.03643495929764)
1,Elbow Drive and Sabrina Road SW,There is an incident involving a pedestrian- EMS on site.,2026/09/18 10:24:28 PM,2026/09/18 10:25:04 PM,SW,-114.084698,50.957010,1.0,2026-09-18 22:24:28+00:0050.95700998524688-114.08469755874496,POINT (-114.08469755874496 50.95700998524688)
2,Northbound Deerfoot Trail approaching 17 Avenue SE,Traffic incident.,2026/09/18 10:35:44 PM,2026/09/18 10:54:26 PM,SE,-114.010146,51.037056,1.0,2026-09-18 22:35:44+00:0051.037056011647046-114.01014602121316,POINT (-114.01014602121316 51.037056011647046)
3,50 Avenue and 1a Street SE,Traffic incident.,2026/09/18 10:23:12 PM,2026/09/18 11:03:21 PM,SE,-114.061387,51.008715,1.0,2026-09-18 22:23:12+00:0051.00871500829836-114.06138745986647,POINT (-114.06138745986647 51.00871500829836)
4,Eastbound Glenmore Trail and Deerfoot Trail SE,Traffic incident. Blocking the left lane,2026/09/18 11:31:40 PM,2026/09/18 11:42:31 PM,SE,-114.039252,50.994171,1.0,2026-09-18 23:31:40+00:0050.994171041580366-114.03925199779404,POINT (-114.03925199779404 50.994171041580366)


In [5]:
# Preview a few traffic-signal records.
display(signals_raw.head())

,INSTDATE,FIRSTROAD,SECONDROAD,QUADRANT,INT_TYPE,PEDBUTTONS,PED_TIMER,ACCESSIBLE_PEDESTRIAN_SIGNAL,latitude,longitude,Point,Count,MISTNO,UNITID
0,2024-09-01,30,SAGE HILL ROW,NW,Pedestrian RRFB,Yes,No,No,51.174647,-114.143124,POINT (-114.14312434101748 51.17464712155771),1,B0128,1110443608
1,2008-02-04,12 AVENUE,3 STREET,SE,Traffic signal,No,Yes,No,51.041231,-114.055927,POINT (-114.05592693181134 51.0412312435667),1,0614,1109001367
2,1968-01-01,SOUTHLAND DRIVE,ELBOW DRIVE,SW,Traffic signal,Yes,Yes,Yes,50.963538,-114.084845,POINT (-114.08484510869329 50.9635377733721),1,0694,1109000993
3,1946-01-01,4 AVENUE,CENTRE STREET,S,Traffic signal,No,Yes,Yes,51.049319,-114.062740,POINT (-114.06274044899592 51.04931908667256),1,0299,1109001284
4,1994-06-10,2 AVENUE,2 STREET,SW,Traffic signal,No,Yes,No,51.051367,-114.067537,POINT (-114.0675373457364 51.05136686461388),1,0253,1109001236


## Initial Data Inspection

Before cleaning, inspect schema, data types, missing values, identifiers, timestamps, coordinates, and categorical fields. This establishes what needs to be changed and what should be preserved.

In [6]:
# Print all source column names before renaming or cleaning them.
print("TRAFFIC INCIDENT COLUMNS")
print("-" * 50)

for column in incidents_raw.columns:
    print(column)

print("\nTRAFFIC SIGNAL COLUMNS")
print("-" * 50)

for column in signals_raw.columns:
    print(column)

TRAFFIC INCIDENT COLUMNS
--------------------------------------------------
INCIDENT INFO
DESCRIPTION
START_DT_UTC
MODIFIED_DT_UTC
QUADRANT
Longitude
Latitude
Count
id
Point

TRAFFIC SIGNAL COLUMNS
--------------------------------------------------
INSTDATE
FIRSTROAD
SECONDROAD
QUADRANT
INT_TYPE
PEDBUTTONS
PED_TIMER
ACCESSIBLE_PEDESTRIAN_SIGNAL
latitude
longitude
Point
Count
MISTNO
UNITID


In [7]:
# Inspect the data type pandas inferred for every source column.
print("INCIDENT DATA TYPES")
display(incidents_raw.dtypes.to_frame("dtype"))

print("\nSIGNAL DATA TYPES")
display(signals_raw.dtypes.to_frame("dtype"))

INCIDENT DATA TYPES


,dtype
INCIDENT INFO,str
DESCRIPTION,str
START_DT_UTC,str
MODIFIED_DT_UTC,str
QUADRANT,str
Longitude,float64
Latitude,float64
Count,float64
id,str
Point,str



SIGNAL DATA TYPES


,dtype
INSTDATE,str
FIRSTROAD,str
SECONDROAD,str
QUADRANT,str
INT_TYPE,str
PEDBUTTONS,str
PED_TIMER,str
ACCESSIBLE_PEDESTRIAN_SIGNAL,str
latitude,float64
longitude,float64


In [8]:
# Count missing values in each incident column and convert the counts to percentages.
incident_missing = (
    incidents_raw.isna()
    .sum()
    .to_frame("missing_count")
)

incident_missing["missing_percent"] = (
    incident_missing["missing_count"]
    / len(incidents_raw)
    * 100
).round(2)

display(incident_missing)


,missing_count,missing_percent
INCIDENT INFO,0,0.00
DESCRIPTION,2,0.00
START_DT_UTC,0,0.00
MODIFIED_DT_UTC,14056,21.69
QUADRANT,14058,21.70
Longitude,0,0.00
Latitude,0,0.00
Count,0,0.00
id,0,0.00
Point,0,0.00


In [9]:
# Count missing values in each signal column and convert the counts to percentages.
signal_missing = (
    signals_raw.isna()
    .sum()
    .to_frame("missing_count")
)

signal_missing["missing_percent"] = (
    signal_missing["missing_count"]
    / len(signals_raw)
    * 100
).round(2)

display(signal_missing)

,missing_count,missing_percent
INSTDATE,0,0.00
FIRSTROAD,0,0.00
SECONDROAD,0,0.00
QUADRANT,0,0.00
INT_TYPE,0,0.00
PEDBUTTONS,34,1.90
PED_TIMER,4,0.22
ACCESSIBLE_PEDESTRIAN_SIGNAL,2,0.11
latitude,0,0.00
longitude,0,0.00


### Duplicate and identifier checks

A unique incident ID and signal ID are important because later spatial joins and summaries assume that each source record represents one distinct observation.

In [10]:
# Check for duplicate rows and duplicate primary identifiers in both datasets.
print("Full duplicate incident rows:",
      incidents_raw.duplicated().sum())

print("Full duplicate signal rows:",
      signals_raw.duplicated().sum())

print()

print("Duplicate incident IDs:",
      incidents_raw["id"].duplicated().sum())

print("Duplicate signal UNITIDs:",
      signals_raw["UNITID"].duplicated().sum())

Full duplicate incident rows: 0
Full duplicate signal rows: 0

Duplicate incident IDs: 0
Duplicate signal UNITIDs: 0


### Timestamp and geographic sanity checks

The incident source stores event times in UTC. We first verify that they parse correctly, then inspect coordinate ranges before converting event times to Calgary local time during cleaning.

In [11]:
# Test whether all incident timestamps parse correctly and inspect the available date range.
incident_dates_test = pd.to_datetime(
    incidents_raw["START_DT_UTC"],
    format="%Y/%m/%d %I:%M:%S %p",
    errors="coerce",
    utc=True
)

print("Unparseable timestamps:",
      incident_dates_test.isna().sum())

print("Earliest incident:",
      incident_dates_test.min())

print("Latest incident:",
      incident_dates_test.max())

Unparseable timestamps: 0
Earliest incident: 2016-12-06 17:00:00+00:00
Latest incident: 2026-09-28 12:35:14+00:00


In [12]:
# Check the minimum and maximum coordinates for obvious geographic data problems.
print("Incident coordinate range")

display(
    incidents_raw[
        ["Latitude", "Longitude"]
    ].agg(["min", "max"])
)

print("\nSignal coordinate range")

display(
    signals_raw[
        ["latitude", "longitude"]
    ].agg(["min", "max"])
)

Incident coordinate range


,Latitude,Longitude
min,50.848554,-114.309770
max,51.198505,-113.865579



Signal coordinate range


,latitude,longitude
min,50.861804,-114.258956
max,51.185747,-113.888654


In [13]:
# Inspect the distribution of incident quadrants, including missing values.
display(
    incidents_raw["QUADRANT"]
    .value_counts(dropna=False)
    .to_frame("count")
)

,count
QUADRANT,
SE,15833
NaN,14058
NE,13749
NW,11076
SW,10082


In [14]:
# Inspect the different types of traffic controls represented in the signal dataset.
display(
    signals_raw["INT_TYPE"]
    .value_counts()
    .to_frame("count")
)

,count
INT_TYPE,
Traffic signal,971
Pedestrian RRFB,270
Traffic signal T intersection,255
Overhead Flasher,255
1/2 signal,22
1/4 signal,11
Fire signal,7


### Data-quality findings

For this downloaded snapshot, the source contains **64,798 incidents** and **1,791 traffic-control records**. There are no duplicate full rows, duplicate incident IDs, or duplicate signal IDs. The incident data has 2 missing descriptions, 14,058 missing quadrants, and 14,056 missing modification timestamps. These fields are not all essential to the analysis, so valid incidents are preserved rather than discarded. The signal data has a small number of missing pedestrian-feature fields, which are also preserved as missing rather than guessed.

## Clean Traffic Incidents

Cleaning decisions:

- preserve the raw dataframe
- rename fields using snake_case
- parse UTC timestamps
- convert event time into Calgary local time
- standardize text whitespace
- preserve missing modified timestamps
- replace missing description/quadrant labels with explicit values
- ensure coordinates are numeric
- retain unique incidents
- remove redundant geometry/count fields

In [15]:
# Work on a copy so the original raw DataFrame remains unchanged.
incidents = incidents_raw.copy()

In [16]:
# Rename source columns to consistent snake_case names for easier use later.
incidents = incidents.rename(
    columns={
        "id": "incident_id",
        "INCIDENT INFO": "incident_location",
        "DESCRIPTION": "description",
        "START_DT_UTC": "start_dt_utc",
        "MODIFIED_DT_UTC": "modified_dt_utc",
        "QUADRANT": "quadrant",
        "Latitude": "latitude",
        "Longitude": "longitude",
    }
)

In [17]:
# Parse the incident and modification timestamps as timezone-aware UTC datetimes.
incidents["start_dt_utc"] = pd.to_datetime(
    incidents["start_dt_utc"],
    format="%Y/%m/%d %I:%M:%S %p",
    errors="coerce",
    utc=True
)

incidents["modified_dt_utc"] = pd.to_datetime(
    incidents["modified_dt_utc"],
    format="%Y/%m/%d %I:%M:%S %p",
    errors="coerce",
    utc=True
)

In [18]:
# Convert UTC event times to Calgary local time; this automatically handles daylight saving time.
incidents["start_dt_local"] = (
    incidents["start_dt_utc"]
    .dt.tz_convert("America/Edmonton")
)

In [19]:
# Helper function: normalize repeated whitespace and remove leading/trailing spaces.
def clean_text(series):
    return (
        series
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [20]:
# Clean incident text fields and use explicit labels where source values are missing.
incidents["incident_location"] = clean_text(
    incidents["incident_location"]
)

incidents["description"] = (
    clean_text(incidents["description"])
    .fillna("Unknown")
)

incidents["quadrant"] = (
    clean_text(incidents["quadrant"])
    .fillna("UNKNOWN")
    .str.upper()
)

In [21]:
# Force latitude and longitude to numeric values; invalid text would become NaN.
incidents["latitude"] = pd.to_numeric(
    incidents["latitude"],
    errors="coerce"
)

incidents["longitude"] = pd.to_numeric(
    incidents["longitude"],
    errors="coerce"
)

In [22]:
# Verify that the fields required for temporal and spatial analysis are present.
incidents[
    [
        "incident_id",
        "start_dt_utc",
        "latitude",
        "longitude"
    ]
].isna().sum()

incident_id     0
start_dt_utc    0
latitude        0
longitude       0
dtype: int64

In [23]:
# Confirm that Count is constant before removing it as a redundant field.
print(incidents["Count"].value_counts())

Count
1.0    64798
Name: count, dtype: int64


In [24]:
# Remove fields that duplicate information already represented by one row and the coordinates.
incidents = incidents.drop(
    columns=["Count", "Point"]
)

## Temporal Feature Engineering

All temporal features are derived from Calgary local time rather than UTC.

In [25]:
# Convert UTC event times to Calgary local time; this automatically handles daylight saving time.
incidents["date_local"] = (
    incidents["start_dt_local"].dt.date
)

incidents["year"] = (
    incidents["start_dt_local"].dt.year
)

incidents["month"] = (
    incidents["start_dt_local"].dt.month
)

incidents["month_name"] = (
    incidents["start_dt_local"].dt.month_name()
)

incidents["day"] = (
    incidents["start_dt_local"].dt.day
)

incidents["weekday"] = (
    incidents["start_dt_local"].dt.day_name()
)

incidents["weekday_number"] = (
    incidents["start_dt_local"].dt.weekday
)

incidents["hour"] = (
    incidents["start_dt_local"].dt.hour
)

incidents["weekend"] = (
    incidents["weekday_number"] >= 5
)

In [26]:
# Map each calendar month to a simple meteorological season.
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"

    if month in [3, 4, 5]:
        return "Spring"

    if month in [6, 7, 8]:
        return "Summer"

    return "Fall"

In [27]:
# Add the season label to every incident.
incidents["season"] = (
    incidents["month"]
    .apply(get_season)
)

In [28]:
# Group hours into project-defined time periods used for rush-hour comparisons.
def get_time_period(hour):
    if 0 <= hour <= 5:
        return "Night"

    if 6 <= hour <= 9:
        return "Morning Rush"

    if 10 <= hour <= 14:
        return "Midday"

    if 15 <= hour <= 18:
        return "Evening Rush"

    return "Evening"

In [29]:
# Add the time-period category to every incident.
incidents["time_period"] = (
    incidents["hour"]
    .apply(get_time_period)
)

In [30]:
# Spot-check the UTC-to-local conversion and the derived temporal features.
display(
    incidents[
        [
            "start_dt_utc",
            "start_dt_local",
            "year",
            "month_name",
            "weekday",
            "hour",
            "weekend",
            "season",
            "time_period",
        ]
    ].head(10)
)

,start_dt_utc,start_dt_local,year,month_name,weekday,hour,weekend,season,time_period
0,2026-09-18 21:36:31+00:00,2026-09-18 15:36:31-06:00,2026,September,Friday,15,False,Fall,Evening Rush
1,2026-09-18 22:24:28+00:00,2026-09-18 16:24:28-06:00,2026,September,Friday,16,False,Fall,Evening Rush
2,2026-09-18 22:35:44+00:00,2026-09-18 16:35:44-06:00,2026,September,Friday,16,False,Fall,Evening Rush
3,2026-09-18 22:23:12+00:00,2026-09-18 16:23:12-06:00,2026,September,Friday,16,False,Fall,Evening Rush
4,2026-09-18 23:31:40+00:00,2026-09-18 17:31:40-06:00,2026,September,Friday,17,False,Fall,Evening Rush
5,2026-09-19 00:07:43+00:00,2026-09-18 18:07:43-06:00,2026,September,Friday,18,False,Fall,Evening Rush
6,2026-09-19 00:34:07+00:00,2026-09-18 18:34:07-06:00,2026,September,Friday,18,False,Fall,Evening Rush
7,2026-09-19 00:37:59+00:00,2026-09-18 18:37:59-06:00,2026,September,Friday,18,False,Fall,Evening Rush
8,2026-09-19 01:46:05+00:00,2026-09-18 19:46:05-06:00,2026,September,Friday,19,False,Fall,Evening
9,2026-09-19 01:56:13+00:00,2026-09-18 19:56:13-06:00,2026,September,Friday,19,False,Fall,Evening


## Clean Traffic Signals

The signal dataset provides controlled intersections and crossings that
will later be linked spatially to traffic incidents.

In [31]:
# Work on a copy so the raw traffic-signal DataFrame remains unchanged.
signals = signals_raw.copy()

In [32]:
# Rename signal columns to consistent snake_case names.
signals = signals.rename(
    columns={
        "UNITID": "signal_id",
        "MISTNO": "mist_no",
        "INSTDATE": "install_date",
        "FIRSTROAD": "first_road",
        "SECONDROAD": "second_road",
        "QUADRANT": "quadrant",
        "INT_TYPE": "control_type",
        "PEDBUTTONS": "ped_buttons",
        "PED_TIMER": "ped_timer",
        "ACCESSIBLE_PEDESTRIAN_SIGNAL":
            "accessible_pedestrian_signal",
    }
)

In [33]:
# Normalize whitespace in road names and categorical signal fields.
signal_text_columns = [
    "mist_no",
    "first_road",
    "second_road",
    "quadrant",
    "control_type",
    "ped_buttons",
    "ped_timer",
    "accessible_pedestrian_signal",
]

for column in signal_text_columns:
    signals[column] = clean_text(
        signals[column]
    )

In [34]:
# Standardize signal quadrant labels while preserving any genuinely unknown value.
signals["quadrant"] = (
    signals["quadrant"]
    .fillna("UNKNOWN")
    .str.upper()
)

The source uses several control types (for example, conventional traffic signals, T-intersection signals, pedestrian RRFBs, and flashers). All are retained at this stage. A later analysis notebook can decide which control types belong in a particular intersection-risk calculation.

In [35]:
# Keep the original installation-date text temporarily so multiple source formats can be parsed.
install_raw = (
    signals["install_date"]
    .astype("string")
    .str.strip()
)

In [36]:
# Parse the normal YYYY-MM-DD installation-date format.
install_date = pd.to_datetime(
    install_raw,
    format="%Y-%m-%d",
    errors="coerce"
)

In [37]:
# Identify the exceptional 14-digit timestamp format found in the source data.
timestamp_mask = (
    install_date.isna()
    & install_raw.str.fullmatch(
        r"\d{14}",
        na=False
    )
)

In [38]:
# Parse the exceptional YYYYMMDDHHMMSS value instead of dropping that record.
install_date.loc[timestamp_mask] = (
    pd.to_datetime(
        install_raw.loc[timestamp_mask],
        format="%Y%m%d%H%M%S",
        errors="coerce"
    )
)

In [39]:
# Replace the raw installation-date strings with the parsed datetime values.
signals["install_date"] = install_date

In [40]:
# Confirm that every installation date was parsed successfully.
print(
    "Unparsed installation dates:",
    signals["install_date"].isna().sum()
)

Unparsed installation dates: 0


In [41]:
# Ensure signal coordinates are numeric for later distance calculations.
signals["latitude"] = pd.to_numeric(
    signals["latitude"],
    errors="coerce"
)

signals["longitude"] = pd.to_numeric(
    signals["longitude"],
    errors="coerce"
)

In [42]:
# Build a readable intersection/control location from the two road-name fields.
signals["intersection_name"] = (
    signals["first_road"]
    + " & "
    + signals["second_road"]
)

In [43]:
# Confirm that Count is constant before removing it.
signals["Count"].value_counts()

Count
1    1791
Name: count, dtype: int64

In [44]:
# Remove redundant Count and Point fields; latitude/longitude already preserve location.
signals = signals.drop(
    columns=["Count", "Point"]
)

## Data Validation

The pipeline should fail visibly if fundamental assumptions about the
cleaned datasets are violated.

In [45]:
# Validate identifiers, timestamps, and geographic fields used by later notebooks.
assert incidents["incident_id"].is_unique
assert signals["signal_id"].is_unique

assert incidents["start_dt_utc"].notna().all()
assert incidents["start_dt_local"].notna().all()

assert incidents["latitude"].notna().all()
assert incidents["longitude"].notna().all()

assert signals["latitude"].notna().all()
assert signals["longitude"].notna().all()

assert incidents["latitude"].between(
    -90, 90
).all()

assert incidents["longitude"].between(
    -180, 180
).all()

assert signals["latitude"].between(
    -90, 90
).all()

assert signals["longitude"].between(
    -180, 180
).all()

print("All validation checks passed.")

All validation checks passed.


In [46]:
# Keep the final incident fields in a stable, readable order and sort chronologically.
incident_columns = [
    "incident_id",
    "incident_location",
    "description",
    "quadrant",
    "start_dt_utc",
    "start_dt_local",
    "modified_dt_utc",
    "date_local",
    "year",
    "month",
    "month_name",
    "day",
    "weekday",
    "weekday_number",
    "hour",
    "weekend",
    "season",
    "time_period",
    "latitude",
    "longitude",
]

incidents = (
    incidents[incident_columns]
    .sort_values("start_dt_utc")
    .reset_index(drop=True)
)

In [47]:
# Keep the final signal fields in a stable order and sort by signal identifier.
signal_columns = [
    "signal_id",
    "mist_no",
    "install_date",
    "first_road",
    "second_road",
    "intersection_name",
    "quadrant",
    "control_type",
    "ped_buttons",
    "ped_timer",
    "accessible_pedestrian_signal",
    "latitude",
    "longitude",
]

signals = (
    signals[signal_columns]
    .sort_values("signal_id")
    .reset_index(drop=True)
)

In [48]:
# Report the final shape of both cleaned datasets.
print(
    f"Clean incidents: "
    f"{len(incidents):,} rows × "
    f"{len(incidents.columns)} columns"
)

print(
    f"Clean signals: "
    f"{len(signals):,} rows × "
    f"{len(signals.columns)} columns"
)

Clean incidents: 64,798 rows × 20 columns
Clean signals: 1,791 rows × 13 columns


In [49]:
# Preview the cleaned incident dataset.
display(incidents.head())

,incident_id,incident_location,description,quadrant,start_dt_utc,start_dt_local,modified_dt_utc,date_local,year,month,month_name,day,weekday,weekday_number,hour,weekend,season,time_period,latitude,longitude
0,2016-12-06 17:00:00+00:00-114.064987451.09611149,Eastbound McKnight Boulevard at 2 Street NW,Multi vehicle incident.,NW,2016-12-06 17:00:00+00:00,2016-12-06 10:00:00-07:00,2016-12-06 17:01:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,10,False,Winter,Midday,51.096111,-114.064987
1,2016-12-06 21:36:00+00:00-114.020547951.0476343,Eastbound Memorial Drive approaching Deerfoot Trail NE,2 vehicle incident blocking the middle lane.,NE,2016-12-06 21:36:00+00:00,2016-12-06 14:36:00-07:00,2016-12-06 21:42:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,14,False,Winter,Midday,51.047634,-114.020548
2,2016-12-06 23:25:00+00:00-114.058178551.04447099,Macleod Trail at 9 Avenue SE,2 vehicle incident.,SE,2016-12-06 23:25:00+00:00,2016-12-06 16:25:00-07:00,2016-12-06 23:26:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,16,False,Winter,Evening Rush,51.044471,-114.058178
3,2016-12-06 23:26:00+00:00-114.030871751.02839263,Ogden Road at Bonnybrook Road SE,2 vehicle incident.,SE,2016-12-06 23:26:00+00:00,2016-12-06 16:26:00-07:00,2016-12-06 23:38:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,16,False,Winter,Evening Rush,51.028393,-114.030872
4,2016-12-07 00:05:00+00:00-114.119583551.06639113,Southbound University Drive at Crowchild Trail NW,2 vehicle incident.,NW,2016-12-07 00:05:00+00:00,2016-12-06 17:05:00-07:00,2016-12-07 00:10:00+00:00,2016-12-06,2016,12,December,6,Tuesday,1,17,False,Winter,Evening Rush,51.066391,-114.119584


In [50]:
# Preview the cleaned traffic-signal dataset.
display(signals.head())

,signal_id,mist_no,install_date,first_road,second_road,intersection_name,quadrant,control_type,ped_buttons,ped_timer,accessible_pedestrian_signal,latitude,longitude
0,1109000525,0909,2000-11-02,CROWCHILD TRAIL,12 MILE COULEE ROAD,CROWCHILD TRAIL & 12 MILE COULEE ROAD,NW,Traffic signal,Yes,Yes,Yes,51.143437,-114.256977
1,1109000526,0892,2006-01-11,TUSCANY BOULEVARD,TUSCANY WAY,TUSCANY BOULEVARD & TUSCANY WAY,NW,Traffic signal,Yes,No,No,51.123965,-114.246059
2,1109000527,0891,2005-08-16,TUSCANY BOULEVARD,TUSCANY VALLEY VIEW,TUSCANY BOULEVARD & TUSCANY VALLEY VIEW,NW,Traffic signal,Yes,No,No,51.124997,-114.242822
3,1109000528,0868,2007-05-07,5029,NOSE HILL DRIVE,5029 & NOSE HILL DRIVE,NW,Traffic signal T intersection,Yes,No,No,51.107086,-114.239813
4,1109000529,0890,2005-10-06,TUSCANY BOULEVARD,TUSCANY SPRINGS BOULEVARD,TUSCANY BOULEVARD & TUSCANY SPRINGS BOULEVARD,NW,Traffic signal T intersection,Yes,No,No,51.125473,-114.238610


In [51]:
# Review remaining incident missing values after cleaning.
display(
    incidents.isna()
    .sum()
    .to_frame("missing")
)

,missing
incident_id,0
incident_location,0
description,0
quadrant,0
start_dt_utc,0
start_dt_local,0
modified_dt_utc,14056
date_local,0
year,0
month,0


In [52]:
# Review remaining signal missing values after cleaning.
display(
    signals.isna()
    .sum()
    .to_frame("missing")
)

,missing
signal_id,0
mist_no,0
install_date,0
first_road,0
second_road,0
intersection_name,0
quadrant,0
control_type,0
ped_buttons,34
ped_timer,4


## Export Cleaned Data

The cleaned files become the stable inputs for subsequent notebooks.
Future analysis notebooks should load these files rather than repeating
the entire cleaning process.

In [53]:
# Create the cleaned-data folder if needed and define the two output file paths.
CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INCIDENTS_CLEAN_FILE = (
    CLEAN_DIR / "incidents_clean.csv"
)

SIGNALS_CLEAN_FILE = (
    CLEAN_DIR / "signals_clean.csv"
)

In [54]:
# Export the cleaned datasets without pandas row-index columns.
incidents.to_csv(
    INCIDENTS_CLEAN_FILE,
    index=False
)

signals.to_csv(
    SIGNALS_CLEAN_FILE,
    index=False
)

print("Saved:")
print(INCIDENTS_CLEAN_FILE)
print(SIGNALS_CLEAN_FILE)

Saved:
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\incidents_clean.csv
c:\Users\ahnaf\Desktop\calgary-traffic-risk-intelligence\data\cleaned\signals_clean.csv


## Data Preparation Summary

This notebook creates two stable, analysis-ready datasets while keeping the raw downloads unchanged.

**Traffic incidents:** timestamps are parsed, converted from UTC to Calgary local time, text fields are standardized, and temporal features such as year, month, weekday, hour, season, and time period are added.

**Traffic signals:** road/control fields are standardized, installation dates are parsed (including the one alternate source format), readable intersection names are created, and coordinates are retained for later spatial matching.

The remaining missing values are intentional source-data gaps: incident modification timestamps and a small number of pedestrian-related signal attributes. The next notebook can use `incidents_clean.csv` and `signals_clean.csv` directly without repeating this cleaning stage.